# Feature Engineering & Feature Selection
- Create new features
- Correlation-based feature selection (for Rating prediction)
- Variance-based feature selection (for Clustering/Segmentation)

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

In [7]:
# Load cleaned data
print("Loading cleaned dataset...")
df = pd.read_csv('phones_cleaned_imputed.csv')
print(f"Shape: {df.shape}\n")

Loading cleaned dataset...
Shape: (1532, 13)



In [8]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================
print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# 1. Price_per_GB_RAM
df['Price_per_GB_RAM'] = df['Price_INR'] / df['RAM_GB']
print("1. Price_per_GB_RAM created")

# 2. Camera_Score
df['Camera_Score'] = df['Rear_Camera_MP'] + df['Front_Camera_MP']
print("2. Camera_Score created")

# 3. Value_Score
df['Value_Score'] = (df['RAM_GB'] * df['Battery_mAh']) / df['Price_INR']
print("3. Value_Score created")

# 4. Price_Tier
df['Price_Tier'] = pd.cut(df['Price_INR'],
                           bins=[0, 4000, 7000, 10000, float('inf')],
                           labels=['Budget', 'Mid', 'Premium', 'Ultra'])
print("4. Price_Tier created")

# 5. Correlation of new features with Rating
new_features = ['Price_per_GB_RAM', 'Camera_Score', 'Value_Score']
print("\nCorrelation of new features with Rating:")
for feat in new_features:
    corr = df[feat].corr(df['Rating'])
    print(f"  {feat}: {corr:.4f}")

print(f"\nDataset shape: {df.shape}")

FEATURE ENGINEERING
1. Price_per_GB_RAM created
2. Camera_Score created
3. Value_Score created
4. Price_Tier created

Correlation of new features with Rating:
  Price_per_GB_RAM: -0.1005
  Camera_Score: 0.2760
  Value_Score: 0.1420

Dataset shape: (1532, 17)


In [9]:
# ============================================================
# FEATURE SELECTION (Simple: Correlation-based)
# ============================================================
print("\n" + "=" * 60)
print("FEATURE SELECTION (Correlation-based)")
print("=" * 60)

# Select numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude = ['Product Name', 'Description', 'Price', 'Rating', 'RAM_Bucket',
           'Cluster', 'Cluster_Label']
feature_cols = [c for c in numeric_cols if c not in exclude]

X = df[feature_cols]
y = df['Rating']

# Correlation with target
print("\nFeature correlation with Rating:")
corr_with_target = X.corrwith(y).abs().sort_values(ascending=False)
print(corr_with_target.round(4))

# Keep features with correlation > 0.05
selected_features = corr_with_target[corr_with_target > 0.05].index.tolist()
print(f"\nFeatures with |corr| > 0.05: {len(selected_features)}")
print(selected_features)

# Check for high inter-feature correlation (drop one of pair if > 0.9)
corr_matrix = X[selected_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
selected_features = [f for f in selected_features if f not in to_drop]
print(f"\nAfter removing highly correlated features: {len(selected_features)}")
print(selected_features)


FEATURE SELECTION (Correlation-based)

Feature correlation with Rating:
Camera_Score        0.2760
Rear_Camera_MP      0.2739
RAM_GB              0.2428
ROM_GB              0.2226
Front_Camera_MP     0.1564
Value_Score         0.1420
Price_INR           0.1274
Screen_cm           0.1236
Price_per_GB_RAM    0.1005
Is_5G               0.0940
Battery_mAh         0.0384
dtype: float64

Features with |corr| > 0.05: 10
['Camera_Score', 'Rear_Camera_MP', 'RAM_GB', 'ROM_GB', 'Front_Camera_MP', 'Value_Score', 'Price_INR', 'Screen_cm', 'Price_per_GB_RAM', 'Is_5G']

After removing highly correlated features: 9
['Camera_Score', 'RAM_GB', 'ROM_GB', 'Front_Camera_MP', 'Value_Score', 'Price_INR', 'Screen_cm', 'Price_per_GB_RAM', 'Is_5G']


In [ ]:
# ============================================================
# FEATURE SELECTION FOR CLUSTERING (Variance-based)
# ============================================================
print("\n" + "=" * 60)
print("FEATURE SELECTION FOR CLUSTERING (Variance-based)")
print("=" * 60)

from sklearn.feature_selection import VarianceThreshold

# Select numeric columns only (exclude text/id/target columns)
cluster_numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cluster_exclude = ['Product Name', 'Description', 'Price', 'Rating', 'RAM_Bucket',
                   'Cluster', 'Cluster_Label']
cluster_candidate_cols = [c for c in cluster_numeric_cols if c not in cluster_exclude]

print(f"Candidate numeric features: {len(cluster_candidate_cols)}")
print(cluster_candidate_cols)

# 1. Variance Threshold: remove near-zero variance features
X_cluster = df[cluster_candidate_cols].fillna(0)

# Scale first so variance is comparable across features
scaler_temp = StandardScaler()
X_scaled_temp = scaler_temp.fit_transform(X_cluster)

variances = pd.Series(np.var(X_scaled_temp, axis=0), index=cluster_candidate_cols)
print(f"\nFeature variances (after scaling):")
print(variances.sort_values(ascending=False).round(4))

# Remove features with variance < 0.1 (near-constant)
var_threshold = 0.1
high_var_features = variances[variances >= var_threshold].index.tolist()
print(f"\nFeatures with variance >= {var_threshold}: {len(high_var_features)}")
print(high_var_features)

# 2. Remove highly correlated pairs (keep one if corr > 0.9)
if len(high_var_features) > 1:
    corr_mat = df[high_var_features].corr().abs()
    upper_tri = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
    to_drop_clust = [col for col in upper_tri.columns if any(upper_tri[col] > 0.9)]
    clustering_features = [f for f in high_var_features if f not in to_drop_clust]
    print(f"\nAfter removing highly correlated pairs: {len(clustering_features)}")
else:
    clustering_features = high_var_features

print(f"\nFinal clustering features: {len(clustering_features)}")
print(clustering_features)

In [ ]:
# Save all outputs
df.to_csv('phones_features_engineered.csv', index=False)
pd.DataFrame({'feature': selected_features}).to_csv('rating_correlation_features.csv', index=False)
pd.DataFrame({'feature': clustering_features}).to_csv('clustering_features.csv', index=False)
print("\nSaved:")
print("  - phones_features_engineered.csv")
print("  - rating_correlation_features.csv (for Rating prediction)")
print("  - clustering_features.csv (for Clustering/Segmentation)")


Saved: phones_features_engineered.csv, final_features.csv
